# Part 2: Can we omit some controls? (R Implementation)

This notebook analyzes a DAG with multiple control variables to determine the minimal sufficient set for estimating the causal effect of X on Y using R.

In [ ]:
# Load required libraries
library(ggplot2)
library(dplyr)
library(dagitty)
library(ggdag)
library(broom)
library(gridExtra)

# Set random seed for reproducibility
set.seed(42)

# Create output directory if it doesn't exist
output_dir <- "../output"
if (!dir.exists(output_dir)) {
  dir.create(output_dir, recursive = TRUE)
}

## DAG Specification

We need to create a DAG with the following causal relationships:
- $X \rightarrow Y$
- $Z_1 \rightarrow X$, $Z_1 \rightarrow Y$
- $Z_2 \rightarrow X$, $Z_2 \rightarrow Y$
- $Z_3 \rightarrow Z_2$, $Z_3 \rightarrow Y$

In [ ]:
# Create and visualize the DAG
dag <- dagify(
  Y ~ X + Z1 + Z2 + Z3,
  X ~ Z1 + Z2,
  Z2 ~ Z3,
  coords = list(
    x = c(Z3 = 0, Z1 = 0, Z2 = 1, X = 2, Y = 3),
    y = c(Z3 = 2, Z1 = 0, Z2 = 1, X = 0.5, Y = 0.5)
  )
)

p_dag <- ggdag(dag) + 
  theme_dag() + 
  ggtitle("DAG: Multiple Controls Analysis")

print(p_dag)
ggsave(file.path(output_dir, "part2_dag_R.png"), p_dag, width = 10, height = 8, dpi = 300)

## Data Simulation

Following the lab convention where each causal arrow represents a unit effect, we simulate:
- $Z_3 = \varepsilon_{Z_3}$
- $Z_1 = \varepsilon_{Z_1}$  
- $Z_2 = Z_3 + \varepsilon_{Z_2}$
- $X = Z_1 + Z_2 + \varepsilon_X$
- $Y = X + Z_1 + Z_2 + Z_3 + \varepsilon_Y$

In [ ]:
# Set sample size
n <- 10000

# Generate exogenous variables (error terms)
eps_Z3 <- rnorm(n, 0, 1)
eps_Z1 <- rnorm(n, 0, 1)
eps_Z2 <- rnorm(n, 0, 1)
eps_X <- rnorm(n, 0, 1)
eps_Y <- rnorm(n, 0, 1)

# Generate endogenous variables following the DAG structure
Z3 <- eps_Z3
Z1 <- eps_Z1
Z2 <- Z3 + eps_Z2  # Z3 -> Z2
X <- Z1 + Z2 + eps_X  # Z1 -> X, Z2 -> X
Y <- X + Z1 + Z2 + Z3 + eps_Y  # X -> Y, Z1 -> Y, Z2 -> Y, Z3 -> Y

# Create DataFrame
df <- data.frame(
  Z3 = Z3,
  Z1 = Z1, 
  Z2 = Z2,
  X = X,
  Y = Y
)

cat("Data simulation completed. Summary statistics:\n")
print(summary(df))
cat("\nTrue causal effect of X on Y: 1.0\n")

## Regression Analysis

We'll run the following regressions and compare their estimates of the X → Y effect:
1. $Y$ vs. $X$
2. $Y$ vs. $X, Z_1$
3. $Y$ vs. $X, Z_2$
4. $Y$ vs. $X, Z_1, Z_2$
5. $Y$ vs. $X, Z_1, Z_2, Z_3$

In [ ]:
# Function to extract coefficient and confidence interval for X
extract_results <- function(model, var_name = "X") {
  coef_summary <- summary(model)
  coef_X <- coef_summary$coefficients[var_name, "Estimate"]
  se_X <- coef_summary$coefficients[var_name, "Std. Error"]
  
  # 99% confidence interval
  t_val <- qt(0.995, df = model$df.residual)  # 99% CI
  conf_lower <- coef_X - t_val * se_X
  conf_upper <- coef_X + t_val * se_X
  
  return(list(
    coefficient = coef_X,
    std_error = se_X,
    conf_lower = conf_lower,
    conf_upper = conf_upper
  ))
}

# Run all regressions
models <- list(
  "X" = lm(Y ~ X, data = df),
  "X, Z1" = lm(Y ~ X + Z1, data = df),
  "X, Z2" = lm(Y ~ X + Z2, data = df),
  "X, Z1, Z2" = lm(Y ~ X + Z1 + Z2, data = df),
  "X, Z1, Z2, Z3" = lm(Y ~ X + Z1 + Z2 + Z3, data = df)
)

# Extract results
results <- data.frame(
  Model = names(models),
  Coefficient = numeric(length(models)),
  Std_Error = numeric(length(models)),
  CI_Lower_99 = numeric(length(models)),
  CI_Upper_99 = numeric(length(models)),
  stringsAsFactors = FALSE
)

for (i in seq_along(models)) {
  res <- extract_results(models[[i]])
  results[i, "Coefficient"] <- res$coefficient
  results[i, "Std_Error"] <- res$std_error
  results[i, "CI_Lower_99"] <- res$conf_lower
  results[i, "CI_Upper_99"] <- res$conf_upper
}

# Display results
cat("Regression Results (Effect of X on Y):\n")
cat(paste(rep("=", 80), collapse = ""), "\n")
for (i in 1:nrow(results)) {
  cat(sprintf("%d. %-15s | Coef: %6.3f | 99%% CI: [%6.3f, %6.3f]\n", 
              i, results$Model[i], results$Coefficient[i], 
              results$CI_Lower_99[i], results$CI_Upper_99[i]))
}
cat(paste(rep("=", 80), collapse = ""), "\n")
cat("True effect: 1.000\n")

In [ ]:
# Create coefficient plot
results$Model_Factor <- factor(results$Model, levels = results$Model)
results$Error_Lower <- results$Coefficient - results$CI_Lower_99
results$Error_Upper <- results$CI_Upper_99 - results$Coefficient

p_coef <- ggplot(results, aes(x = Model_Factor, y = Coefficient)) +
  geom_point(size = 3) +
  geom_errorbar(aes(ymin = CI_Lower_99, ymax = CI_Upper_99), 
                width = 0.2, size = 1) +
  geom_hline(yintercept = 1.0, color = "red", linetype = "dashed", size = 1) +
  labs(
    x = "Regression Models",
    y = "Estimated Effect of X on Y",
    title = "Coefficient Estimates with 99% Confidence Intervals"
  ) +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
  annotate("text", x = 1, y = 1.05, label = "True Effect = 1.0", color = "red")

print(p_coef)
ggsave(file.path(output_dir, "part2_coefficients_R.png"), p_coef, width = 12, height = 8, dpi = 300)

## Analysis of Results

In [ ]:
# Create results summary table
results$Bias <- results$Coefficient - 1.0
results$Contains_True <- (results$CI_Lower_99 <= 1.0) & (1.0 <= results$CI_Upper_99)

cat("\nDetailed Results Summary:\n")
print(round(results[, c("Model", "Coefficient", "Std_Error", "CI_Lower_99", "CI_Upper_99", "Bias", "Contains_True")], 4))

# Save results
write.csv(results, file.path(output_dir, "part2_regression_results_R.csv"), row.names = FALSE)

In [ ]:
# Print detailed summaries for models 4 and 5
cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat("DETAILED SUMMARY: Model 4 (Y vs X, Z1, Z2)\n")
cat(paste(rep("=", 80), collapse = ""), "\n")
print(summary(models[["X, Z1, Z2"]]))

cat("\n", paste(rep("=", 80), collapse = ""), "\n")
cat("DETAILED SUMMARY: Model 5 (Y vs X, Z1, Z2, Z3)\n")
cat(paste(rep("=", 80), collapse = ""), "\n")
print(summary(models[["X, Z1, Z2, Z3"]]))

## Answers to Questions

### Which regressions seem to estimate the effect correctly?

Based on the results above, we can analyze which regressions provide unbiased estimates of the true causal effect (1.0):

- **Model 1 (X only)**: Biased due to omitted variable bias from Z1 and Z2
- **Model 2 (X, Z1)**: Still biased due to omitting Z2 
- **Model 3 (X, Z2)**: Still biased due to omitting Z1
- **Model 4 (X, Z1, Z2)**: Should provide unbiased estimate - controls for all confounders
- **Model 5 (X, Z1, Z2, Z3)**: May have slightly higher variance due to overcontrolling

### Comment on point estimate and precision for models 4 and 5

Looking at the detailed summaries:
- **Point estimates**: Both should be close to 1.0 (the true effect)
- **Precision**: Model 4 likely has better precision (lower standard errors) as it doesn't include the unnecessary control Z3
- **Model 5** includes Z3 which is not necessary for identification, potentially reducing precision

### Can you ignore some Z ∈ {Z1, Z2, Z3} and get a good estimate?

**Answer**: You can ignore Z3 and still get a good estimate. Here's why:

- **Z1 and Z2 are confounders** - they affect both X and Y, so controlling for them is necessary
- **Z3 is not a confounder of X and Y** - while Z3 affects Y, it doesn't directly affect X (only through Z2)
- **Controlling for Z2 blocks the backdoor path through Z3**
- **The minimal sufficient set is {Z1, Z2}** - this blocks all backdoor paths from X to Y

The backdoor paths from X to Y are:
1. X ← Z1 → Y (blocked by controlling for Z1)
2. X ← Z2 → Y (blocked by controlling for Z2)  
3. X ← Z2 ← Z3 → Y (blocked by controlling for Z2)

Therefore, **Model 4 (X, Z1, Z2) provides the optimal balance** of unbiased estimation with maximum precision.

In [ ]:
# Save the simulated data
write.csv(df, file.path(output_dir, "part2_simulated_data_R.csv"), row.names = FALSE)

cat("\nAll results saved to output directory.\n")
cat("Files created (R implementation):\n")
cat("- part2_dag_R.png\n")
cat("- part2_coefficients_R.png\n")
cat("- part2_regression_results_R.csv\n")
cat("- part2_simulated_data_R.csv\n")